<a href="https://colab.research.google.com/github/hvs515/Federated-Learning-IID-simulation/blob/main/FedIID.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas
import numpy as np
import matplotlib.pyplot as plt
import copy
import random
import torch
import torchvision
from torch.utils.data import DataLoader,Subset
import torchvision.transforms as transforms
from torch import nn, optim

BATCH_SIZE = 32
SEED = 42
TOTAL_CLIENTS=100
CLIENTS_PER_ROUND = 10
COMMUNICATION_ROUNDS = 30
LOCAL_EPOCHS = 5
LEARNING_RATE = 0.01
MOMENTUM = 0.9
NUM_CLASSES = 10

def set_seed(seed):
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  np.random.seed(seed)
  random.seed(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")




In [ ]:

def create_iid_clients(train_dataset, num_clients, batch_size, seed):

    np.random.seed(seed)


    num_samples = len(train_dataset)

    samples_per_client = num_samples // num_clients


    all_indices = list(range(num_samples))

    np.random.shuffle(all_indices)

    client_dataloaders = []

    for i in range(num_clients):

        start_idx = i * samples_per_client
        end_idx = (i + 1) * samples_per_client

        client_indices = all_indices[start_idx:end_idx]


        client_dataset = Subset(train_dataset, client_indices)

        client_dataloader = DataLoader(client_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        client_dataloaders.append(client_dataloader)

    return client_dataloaders

cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std = (0.2471, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cifar10_mean, cifar10_std),
])


train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

num_samples = len(train_dataset)
samples_per_client = num_samples // TOTAL_CLIENTS

client_dataloaders = create_iid_clients(train_dataset, TOTAL_CLIENTS, BATCH_SIZE, SEED)

test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("\nVerifying IID client data distribution for 3 random clients:")
random_client_ids = random.sample(range(TOTAL_CLIENTS), 3)

for client_id in random_client_ids:
    print(f"\nClient {client_id} Class Distribution:")
    dataloader = client_dataloaders[client_id]
    class_counts = {i: 0 for i in range(NUM_CLASSES)}
    total_samples = 0

    for _, labels in dataloader:
        for label in labels:
            class_counts[label.item()] += 1
        total_samples += len(labels)


    for class_id, count in class_counts.items():
        percentage = (count / total_samples) * 100 if total_samples > 0 else 0
        print(f"  Class {class_id}: {count} samples ({percentage:.2f}%) - {(count / samples_per_client) * 100 if samples_per_client > 0 else 0:.2f}% of client's total samples")
    print(f"  Total samples for client {client_id}: {total_samples}")


In [ ]:
import torchvision.models as models
def get_model():
  model = models.resnet18(pretrained=False) #
  num_ftrs = model.fc.in_features
  model.fc = nn.Linear(num_ftrs, NUM_CLASSES)


  model = model.to(device)
  return model

def get_model_weights(model):
  model.load_state_dict(weights)
global_model = get_model()
print(f"Global model (ResNet-18) initialized and moved to {device}. Number of classes in final layer: {global_model.fc.out_features}")
print("Model architecture snippet:")
print(global_model)



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


NameError: name 'nn' is not defined